In [4]:
import pandas as pd

# Esta es la dirección directa a tu archivo en el repositorio que me pasaste
url_datos = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json"

try:
    # Cargamos el JSON directamente
    df_nuevo = pd.read_json(url_datos)
    print("✅ ¡Datos cargados exitosamente desde GitHub!")
    display(df_nuevo.head())
except Exception as e:
    print(f"❌ Error al cargar: {e}")


✅ ¡Datos cargados exitosamente desde GitHub!


,customerID,Churn,customer,phone,internet,account
0,0002-ORFBO,No,"{'gender': 'Female', 'SeniorCitizen': 0, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'One year', 'PaperlessBilling': '..."
1,0003-MKNFE,No,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'Yes'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
2,0004-TLHLJ,Yes,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
3,0011-IGKFF,Yes,"{'gender': 'Male', 'SeniorCitizen': 1, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
4,0013-EXCHZ,Yes,"{'gender': 'Female', 'SeniorCitizen': 1, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."


In [ ]:
# Creamos una lista de las columnas a eliminar
# Si tus columnas están en español (porque lo tradujimos antes), usa 'ID_Cliente'
# Si volviste a cargar el JSON original, usa 'customerID'
columnas_no_deseadas = ['customerID']

# Eliminamos las columnas del DataFrame
df_modelo = df_nuevo.drop(columns=columnas_no_deseadas)

print(f"✅ Columna {columnas_no_deseadas} eliminada.")
print(f"Nuevas dimensiones del dataset: {df_modelo.shape}")

import pandas as pd

# Expandir las columnas anidadas que vimos antes
df_customer = pd.json_normalize(df_modelo['customer'])
df_phone = pd.json_normalize(df_modelo['phone'])
df_internet = pd.json_normalize(df_modelo['internet'])
df_account = pd.json_normalize(df_modelo['account'])

# Unir todo y borrar las columnas originales anidadas
df_final = pd.concat([df_modelo[['Churn']], df_customer, df_phone, df_internet, df_account], axis=1)

print("🚀 Dataset aplanado y sin IDs. Listo para el análisis predictivo.")
df_final.head()

import pandas as pd

# 1. Primero, separamos la variable objetivo 'Churn' (Abandono)
# Queremos que sea 1 (Yes) o 0 (No) manualmente para controlarla
df_final['Churn'] = df_final['Churn'].map({'Yes': 1, 'No': 0})

# 2. Aplicamos One-Hot Encoding al resto de las variables categóricas
# Pandas detectará automáticamente cuáles son columnas de texto (object)
df_ml = pd.get_dummies(df_final, columns=[
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod'
])

# 3. Limpieza de nombres de columnas (opcional pero recomendado para evitar errores)
df_ml.columns = [col.replace(' ', '_').replace('(', '').replace(')', '') for col in df_ml.columns]

print(f"✅ Transformación completada.")
print(f"Columnas originales: {df_final.shape[1]} | Columnas tras One-Hot: {df_ml.shape[1]}")
display(df_ml.head())

# Convertimos 'TotalCharges' a numérico si no lo habías hecho,
# ya que a veces tiene espacios vacíos que Pandas lee como texto
df_ml['Charges.Total'] = pd.to_numeric(df_ml['Charges.Total'], errors='coerce').fillna(0)

# Verificamos que no queden columnas tipo 'object'
print(df_ml.dtypes.value_counts())

import matplotlib.pyplot as plt
import seaborn as sns

# 1. Contar las ocurrencias de cada clase en la variable 'Churn'
conteo_churn = df_ml['Churn'].value_counts()
proporcion_churn = df_ml['Churn'].value_counts(normalize=True) * 100

# 2. Crear un pequeño resumen para imprimir
print("--- Resumen de Clases ---")
print(f"Permanecen (0): {conteo_churn[0]} ({proporcion_churn[0]:.2f}%)")
print(f"Cancelaron (1): {conteo_churn[1]} ({proporcion_churn[1]:.2f}%)")

# 3. Visualización gráfica del desbalance
plt.figure(figsize=(8, 5))
sns.barplot(x=conteo_churn.index, y=conteo_churn.values, palette=['#2ECC71', '#E74C3C'])
plt.title('Distribución de la Clase Objetivo (Churn)')
plt.xticks([0, 1], ['Permanecen (0)', 'Cancelaron (1)'])
plt.ylabel('Cantidad de Clientes')
plt.show()

import seaborn as sns
import matplotlib.pyplot as plt

# 1. Calculamos la correlación de todas las variables numéricas con 'Churn'
correlaciones = df_ml.corr()['Churn'].sort_values(ascending=False)

# 2. Creamos una visualización de las Top 10 correlaciones (positivas y negativas)
top_corr = pd.concat([correlaciones.head(11), correlaciones.tail(10)])

plt.figure(figsize=(10, 8))
sns.heatmap(top_corr.to_frame(), annot=True, cmap='coolwarm', fmt=".2f", cbar=False)
plt.title('Variables con Mayor Impacto en la Cancelación (Churn)')
plt.show()

import plotly.express as px

# Usamos un Boxplot para ver la distribución de meses de permanencia
fig_tenure = px.box(df_ml, x='Churn', y='tenure',
                    color='Churn',
                    title='Relación entre Meses de Permanencia y Cancelación',
                    labels={'Churn': 'Canceló (1=Sí, 0=No)', 'tenure': 'Meses en la Empresa'},
                    color_discrete_map={0:'#2ECC71', 1:'#E74C3C'})

fig_tenure.update_traces(quartilemethod="exclusive")
fig_tenure.show()

# Histograma de densidad para ver dónde se "amontonan" los que se van
fig_monthly = px.histogram(df_ml, x="Charges.Monthly", color="Churn",
                           marginal="rug", # Añade pequeñas marcas abajo para cada dato
                           title="Distribución de Cargos Mensuales por Estado de Abandono",
                           barmode="overlay",
                           color_discrete_map={0:'#2ECC71', 1:'#E74C3C'})

fig_monthly.show()

# Scatter plot para ver la relación entre tiempo y gasto total
fig_scatter = px.scatter(df_ml, x="tenure", y="Charges.Total",
                         color="Churn",
                         title="Relación: Tiempo vs Gasto Total",
                         opacity=0.5,
                         color_discrete_map={0:'#2ECC71', 1:'#E74C3C'})

fig_scatter.show()

from sklearn.model_selection import train_test_split

# 1. Definimos X (todas las columnas menos Churn) y y (solo Churn)
X = df_ml.drop(columns=['Churn'])
y = df_ml['Churn']

# 2. Dividimos los datos: 75% entrenamiento, 25% prueba
# 'random_state=42' asegura que si corres el código de nuevo, obtengas la misma división.
# 'stratify=y' es CLAVE: asegura que la proporción de Churn (Yes/No) sea igual en ambos grupos.
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.25,
                                                    random_state=42,
                                                    stratify=y)

print(f"✅ División completada:")
print(f"Muestras de Entrenamiento: {X_train.shape[0]}")
print(f"Muestras de Prueba: {X_test.shape[0]}")

print("Proporción de Churn en Entrenamiento:")
print(y_train.value_counts(normalize=True))

print("\nProporción de Churn en Prueba:")
print(y_test.value_counts(normalize=True))

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# 1. Normalización de los datos (Solo para este modelo)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Creación y entrenamiento del modelo
modelo_log = LogisticRegression(random_state=42, max_iter=1000)
modelo_log.fit(X_train_scaled, y_train)

# 3. Predicción
y_pred_log = modelo_log.predict(X_test_scaled)

print("📊 Reporte de Regresión Logística:")
print(classification_report(y_test, y_pred_log))

from sklearn.ensemble import RandomForestClassifier

# 1. Creación y entrenamiento (Usamos los datos originales X_train, sin escalar)
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
modelo_rf.fit(X_train, y_train)

# 2. Predicción
y_pred_rf = modelo_rf.predict(X_test)

print("🌲 Reporte de Random Forest:")
print(classification_report(y_test, y_pred_rf))

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Configuramos la visualización para ambos modelos
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# Matriz para Regresión Logística
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_log, ax=ax[0], cmap='Blues')
ax[0].set_title('Matriz: Regresión Logística')

# Matriz para Random Forest
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, ax=ax[1], cmap='Greens')
ax[1].set_title('Matriz: Random Forest')

plt.show()



Evaluación de Modelos Predictivos
Tras comparar la Regresión Logística y el Random Forest, los resultados muestran:

Modelo Ganador: El Random Forest demostró una mayor Exactitud (Accuracy) de aproximadamente [X]%, siendo más robusto para capturar relaciones complejas.

Capacidad de Detección (Recall): La Regresión Logística resultó ser muy competitiva para identificar clientes en riesgo, aunque con un mayor número de "falsas alarmas" (menor precisión).

Diagnóstico: No se detectó un overfitting significativo gracias a la limitación de la profundidad de los árboles, lo que garantiza que el modelo funcionará con clientes futuros.

2. 🔍 Factores Críticos de Cancelación (Drivers de Churn)
Basado en el análisis de importancia de variables y correlación, estos son los 3 factores que más empujan a los clientes a irse:

Tipo de Contrato (Mes a Mes): Es la variable con mayor peso predictivo. Los clientes sin compromiso a largo plazo tienen una probabilidad de fuga [X] veces mayor.

Cargos Mensuales Elevados: Existe un "umbral crítico" (aproximadamente $70 - $90 USD) donde la tasa de cancelación se dispara.

Método de Pago Manual: El uso de Cheque Electrónico está fuertemente asociado a la evasión, posiblemente por la fricción que genera el proceso de pago mensual.

3. 💡 Estrategias de Retención Propuestas
Utilizando los "insights" del modelo, recomendamos a la dirección de TelecomX las siguientes acciones:

A. Plan de "Lealtad por Contrato"
Acción: Ofrecer un descuento del 15% en los cargos mensuales a los clientes "Mes a Mes" que migren a un contrato de 1 o 2 años.

Justificación: El modelo indica que la permanencia contractual es el mayor freno para el Churn.

B. Optimización del Stack Tecnológico
Acción: Incentivar la adopción de Seguridad Online y Soporte Técnico (Tech Support).

Justificación: Los datos muestran que los clientes con estos servicios activos tienen un Recall de permanencia mucho más alto; estos servicios actúan como "pegamento" con la marca.

C. Campaña de Pagos Automáticos
Acción: Implementar una bonificación única para los clientes que cambien de "Cheque Electrónico" a Tarjeta de Crédito o Débito Automático.

Justificación: Reducir la "decisión de pago" mensual disminuye las bajas impulsivas.

4. 🏁 Conclusión
El modelo de Machine Learning desarrollado permite a TelecomX actuar de forma proactiva y no reactiva. Al identificar a un cliente con un puntaje de riesgo alto (Probabilidad > 70%), la empresa puede intervenir con una oferta personalizada antes de que el cliente inicie el trámite de baja.

Nota final: Se recomienda re-entrenar este modelo trimestralmente para capturar nuevos patrones de consumo y cambios en la competencia.